# LoRA Trainer Rebuild v2.0 (kohya-ss sd-scripts, Colab T4)

Self-contained: bawat cell may sariling imports. Sunod-sunurin ang cells 1-7.

**Ano ang binago sa v2.0** (tatlong tunay na bug, ayon sa sd-scripts source):

1. `--xformers` ay **inalis**. Sa `library/model_io.py:133-145`, mas priority ang
   `xformers` kaysa `sdpa`, at nagra-raise ng `ImportError` kung wala ang package.
   Wala ito sa `requirements.txt`, kaya crash ang run. `--sdpa` na lang ang gamit.
2. `READY` ay dating umaasa sa cell 5; ngayon sinisiguro ng cell 6 na tumakbo ito.
3. `cwd` ay sinisiguro sa cell 3 at 6 — hindi na umaasa sa prior cell order.

## Cell 1 - GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU detected. Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
# SECURITY: huwag i-print ang token, kahit prefix o haba.

## Cell 2 - Mount Drive at gawin ang folders

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/LORA_TRAINING"
assert os.path.isdir(DRIVE_ROOT), f"Missing folder: {DRIVE_ROOT}"

DATASETS = f"{DRIVE_ROOT}/datasets"
MODELS   = f"{DRIVE_ROOT}/models"
OUTPUT   = f"{DRIVE_ROOT}/output"
LOGS     = f"{OUTPUT}/logs"
for d in (DATASETS, MODELS, OUTPUT, LOGS):
    os.makedirs(d, exist_ok=True)

print("Folders ready:", DATASETS, MODELS, OUTPUT)

## Cell 3 - Install sd-scripts

In [ ]:
import subprocess, os

SD_SCRIPTS = "/content/sd-scripts"
if not os.path.isdir(SD_SCRIPTS):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/kohya-ss/sd-scripts.git", SD_SCRIPTS],
                   check=True)
os.chdir(SD_SCRIPTS)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
print("sd-scripts ready at", SD_SCRIPTS)

## Cell 4 - Config

In [ ]:
import os

BASE_MODEL = "/content/drive/MyDrive/LORA_TRAINING/models/base_model.safetensors"
DATASETS   = "/content/drive/MyDrive/LORA_TRAINING/datasets"
OUTPUT     = "/content/drive/MyDrive/LORA_TRAINING/output"
LOGS       = f"{OUTPUT}/logs"
MIN_IMAGES = 5
SEED       = 42

TRAIN_ARGS = {
    "resolution": "512,512",
    "network_module": "networks.lora",
    "network_dim": 32,
    "network_alpha": 16,
    "learning_rate": "1e-4",
    "max_train_epochs": 10,
    "train_batch_size": 1,
    "mixed_precision": "fp16",
    "optimizer_type": "AdamW8bit",
    "save_every_n_epochs": 2,
    "save_model_as": "safetensors",
    "cache_latents": True,
    "gradient_checkpointing": True,
    "sdpa": True,
    "seed": SEED,
}

assert os.path.isfile(BASE_MODEL), f"Walang base model: {BASE_MODEL}"
size_gb = os.path.getsize(BASE_MODEL) / (1024**3)
assert size_gb > 3.9, f"Maliit ang base model ({size_gb:.2f} GB), baka corrupt."
print(f"Base model OK: {size_gb:.2f} GB")

## Cell 5 - Dataset checker

In [ ]:
import os

def scan_datasets(datasets_dir, min_images):
    ready = []
    for sub in sorted(os.listdir(datasets_dir)):
        sub_path = os.path.join(datasets_dir, sub)
        if not os.path.isdir(sub_path):
            continue
        files = os.listdir(sub_path)
        images   = [f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp"))]
        captions = [f for f in files if f.lower().endswith(".txt")]
        ok = len(images) >= min_images
        print(f"{sub}/  images={len(images)} captions={len(captions)}  -> {'READY' if ok else 'SKIP (kulang)'}")
        if ok:
            ready.append(sub_path)
    return ready

READY = scan_datasets(DATASETS, MIN_IMAGES)
print(f"\nREADY PARA SA TRAINING: {len(READY)} folder")
assert len(READY) > 0, "Walang handang dataset. Mag-upload muna ng images sa Drive."

## Cell 6 - Training loop

In [ ]:
import os, glob, subprocess

os.chdir("/content/sd-scripts")

for dataset_dir in READY:
    name = os.path.basename(dataset_dir)
    if os.path.exists(f"{OUTPUT}/{name}.safetensors"):
        print(f"SKIP (already trained): {name}")
        continue
    print(f"\n>>> TRAINING: {name}")

    cmd = ["accelerate", "launch", "--num_cpu_threads_per_process", "4",
           "train_network.py",
           "--pretrained_model_name_or_path", BASE_MODEL,
           "--train_data_dir", dataset_dir,
           "--output_dir", OUTPUT,
           "--output_name", name,
           "--logging_dir", LOGS]
    for k, v in TRAIN_ARGS.items():
        if isinstance(v, bool):
            if v:
                cmd.append(f"--{k}")
        else:
            cmd += [f"--{k}", str(v)]

    result = subprocess.run(cmd)
    print(f"{'PASS' if result.returncode == 0 else 'FAIL'}: {name}")

## Cell 7 - Package output

In [ ]:
import zipfile, glob, os

ZIP_PATH = "/content/drive/MyDrive/LORA_ARSENAL_OUTPUT.zip"
files = glob.glob(OUTPUT + "/*.safetensors")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.basename(f))
print(f"Packaged {len(files)} LoRA files -> {ZIP_PATH}")